In [61]:
from pathlib import Path
import re

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

import pandas as pd
import numpy as np
from transformers import AutoModel, AutoTokenizer, AutoConfig

import src
from scipy.special import softmax

In [4]:
pd.set_option("display.max_colwidth", 512)

In [5]:
nltk.download("vader_lexicon", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

True

In [87]:
from transformers import pipeline

model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

pipe = pipeline("text-classification", model_name)
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [ ]:
analyzer = SentimentIntensityAnalyzer()


STOPWORDS = set(stopwords.words("english"))


def tokenize(text: str) -> list[str]:
    return word_tokenize(text, language="english")


def clean(text: str, exclude_too: str):
    # unify text
    text = text.strip().lower().replace("\n", " ")

    # remove urls
    text = re.sub(r"(?:\@|http?\://|https?\://|www)\S+", "", text)

    # remove non-words
    text = re.sub(r"[^\w\s]+|\d+|#\S+", "", text)

    # remove additional text (search query)
    text = text.replace(exclude_too, "")

    # remove stopwords
    tokens = tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS]

    return text


def analyze(analyzer, text: str, exclude_too: str):
    clean_text = clean(text, exclude_too)
    scores = analyzer.polarity_scores(clean_text)
    return scores["compound"]

In [46]:
node_folder = src.PATH / "data/interim/node_lists/"
node_files = list(node_folder.iterdir())

In [99]:
in_file = node_files[15]
in_file

PosixPath('/Users/lukas/git/yt_rabbit_hole/data/interim/node_lists/5g_covid.csv')

In [101]:
df = pd.read_csv(in_file)[["title", "description"]]
df.head()

,title,description
0,Does 5G cause coronavirus?,Subscribe: http://bit.ly/WeeklyYouTube\nFacebook: http://bit.ly/WeeklyFacebook \nTwitter: http://bit.ly/WeeklyTwitter
1,The Facts Behind 5G &amp; Coronavirus | Mashable Explains,"5G has been blamed for increased rates of cancer, autism, and even infertility. Now it's being blamed for the coronavirus pandemic. Here's why none of this is true.\n\nMashable is your source for the latest in tech, culture, and entertainment. \nSubscribe to Mashable: https://bit.ly/2DR64oM\nWatch more episodes of Mashable Explains: http://bit.ly/36y6pcE\n\nFollow us:\nCheck out www.mashable.com\nFacebook: http://on.mash.to/2lyOwmZ\nTwitter: http://on.mash.to/1Udp1kz\nInstagram: http://on.mash.to/1U6D4..."
2,Inside COVID-19 conspiracy theories: from 5G towers to Bill Gates | 60 Minutes Australia,"Subscribe here: http://9Soci.al/chmP50wA97J Full Episodes: https://9now.app.link/uNP4qBkmN6 | Mad as hell (2020)\n\nJust who is hoodwinking who? There’s no doubt COVID-19 has caused great uncertainty in the world, but does that mean we should now ignore the scientists, doctors and even politicians who are fighting to figure out ways to beat the virus? Well yes, if you believe an increasing number of increasingly angry people who are convinced coronavirus is nothing more than a sinister plot to control t..."
3,Why the 5G coronavirus conspiracy theory is false,"Conspiracy theories linking 5G technology to coronavirus have resulted in dozens of phone masts across the UK being vandalised in recent weeks. Theories about the dangers of 5G had already been circulating, despite regulators confirming that the radiation levels of the new technology are well within safe boundaries. So how did the conspiracy incorrectly linking it to 5G start? And is 5G really dangerous? We explain why 5G has nothing to do with Covid-19\n\nSubscribe to The Guardian on YouTube ► http://i..."
4,"Coronavirus outbreak: Conspiracy theorists burn 5G towers, claiming link to COVID-19","Some conspiracy theorists are falsely linking 5G, the fifth generation of wireless mobile technology, to COVID-19 outbreaks. Jeff Semple looks at what started this hoax, how it spread like wildfire on social media and how it ignited attacks on cellphone towers.\n\nFor more info, please go to https://globalnews.ca/news/6915110/quebec-cellphone-tower-fires-arrests/\n\nSubscribe to Global News Channel HERE: http://bit.ly/20fcXDc\r\nLike Global News on Facebook HERE: http://bit.ly/255GMJQ\r\nFollow Global N..."


In [103]:
def roberta_sentiment(text, exclude_too):
    global model, tokenizer, config
    if pd.isnull(text):
        return (np.nan, np.nan)
    text = clean(text, exclude_too)
    out = pipe(text, truncation=True, max_length=512)
    out = out[0]
    return out["label"], out["score"]

In [104]:
search_query = in_file.name.replace("_", " ").replace(".csv", "")
df["sentiment_title_vader"] = df["title"].apply(
    lambda x: analyze(analyzer, x, search_query)
)
df["sentiment_title_roberta_label"], df["sentiment_title_roberta_score"] = zip(
    *df["title"].apply(lambda x: roberta_sentiment(x, search_query))
)

df["sentiment_desc_roberta_label"], df["sentiment_desc_roberta_score"] = zip(
    *df["description"].apply(lambda x: roberta_sentiment(x, search_query))
)

In [10]:
out_folder = src.PATH / "data/interim/title_sentiments/"
out_folder.mkdir(exist_ok=True, parents=True)

In [ ]:
for file in node_files:
    filename = file.name
    search_query = file.name.replace("_", " ").replace(".csv", "")
    out_file = out_folder / filename

    df = pd.read_csv(file)

    df[["video_id", "sentiment"]].to_csv(out_file, index=False)